# A5 — `C.U40k.rsmiles10.e2 → Ut`: 10 renderings per reaction

A4 established that the two techniques stack rather than overlap: root-alignment alone reached
69.5% exact top-5, online randomization alone 70.9%, and the two combined as the published
R-SMILES recipe (5 renderings, both sides rooted consistently) 76.8% — significantly above each
component (p<0.0001 against both). This run pushes the one axis A4 left unexplored: how many
renderings. The published recipe uses 20; A4 used 5; this uses 10.

Two epochs rather than three keeps the sample count comparable to a longer run over fewer
renderings (793k against A4's 595k) while every sample is a distinct rendering rather than a
repeat — the direction the R-SMILES ablation reports as the one that matters.

**Reference points on the same 1000-record comparison set:** A0 (untuned CompoundT5) 0.0% exact
with 14.5% parseable SMILES; A1 canonical 39.3/56.9/63.7; A2 root-aligned 44.2/62.9/69.5;
A3 augmented 43.8/64.3/70.9; A4 R-SMILES x5 48.4/69.9/76.8; ensemble A4+A3 48.4/70.6/78.2.

**Benchmark.** `bisectgroup/USPTO_50K`, canonical 40008/5001/5007 split. Training rows from
`train`, test from `test` (5003 unique products, 82 overlapping rows dropped from the training
side, so the overlap is exactly zero). `sagawa/CompoundT5` was pretrained by span-MLM over 24M
ZINC20 molecules and has never seen a reaction, so nothing here can leak through pretraining.
Literature on this split: R-SMILES 56.3% top-1 / 86.2% top-5; RetroKNN 57.2% top-1.

**Data:** `kuzmenkooleh/retro-planner-uspto50k-rsmiles10` (396,589 train + 4,986 val).

**Cost:** ~7 h training, then both evaluations — the 1000-record comparison set (~35 min) and the
full 5003-record test (~2 h 50 min), which is the number the thesis reports. Total stays under
Kaggle's 12 h session cap.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4
num_train_epochs = 2
output_dir = "/kaggle/working/model1_compoundt5_uspto_rsmiles10"
time_budget_minutes = 420

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs {num_train_epochs} \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The vocabulary repair must have fired (CompoundT5 ships a 221-token ZINC vocabulary with
# no `.` separator). If this line is absent the run trained on <unk>-corrupted targets.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

In [ ]:
# Whether eval_loss was still falling at the end decides what a further run would change:
# A1/A2 turned up at epoch 3.8, A3 was still falling at epoch 9.9.
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
# 1000-record set first (comparable to A0-A4), then the full 5003-record test.
model_dir = f"{output_dir}/final"
for tag, targets in [("uspto1000", "data/v2_uspto_test_holdout_1000.json"),
                     ("uspto5003", "data/v2_uspto_test_holdout.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/A5_rsmiles10_e2_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("uspto1000", "uspto5003"):
    data = json.load(open(f"/kaggle/working/A5_rsmiles10_e2_{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data["summary"], indent=2))